# Voice Similarity Checker

Audio biometrics tool that determines whether two voice recordings belong to the same person, using **MFCC cosine distance** and spectral feature analysis.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DaniAngel79/voice-similarity-checker/blob/main/voice_similarity_checker.ipynb)

---
## How it works
1. Load two audio files (WAV or MP3)
2. Extract **13 MFCC coefficients** per audio (mean over time)
3. Compute **cosine distance** between MFCC vectors
4. Extract spectral features: centroid, flatness, rolloff
5. Verdict: MFCC distance < 0.15 → same person
6. Save spectrogram image + PDF report


In [3]:
!apt-get install -y ffmpeg -q
!pip install librosa gradio fpdf2 soundfile -q


E: No se pudo abrir el fichero de bloqueo «/var/lib/dpkg/lock-frontend» - open (13: Permiso denegado)
E: No se pudo obtener el bloqueo de la interfaz dpkg (/var/lib/dpkg/lock-frontend). ¿Es usted superusuario?
error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try apt install
    python3-xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Debian-packaged Python package,
    create a virtual environment using python3 -m venv path/to/venv.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip. Make
    sure you have python3-full installed.
    
    If you wish to install a non-Debian packaged Python application,
    it may be easiest to use pipx install xyz, which will manage a
    virtual environment for you. Make sure you have pipx installed.
    
    See /usr/share/doc/python3.12/README.venv for more information.

note: If you believe this is a mistake, pleas

In [2]:
import librosa
import numpy as np
import matplotlib.pyplot as plt
from fpdf import FPDF
import gradio as gr
import subprocess, os, tempfile


def convert_to_wav(input_path):
    """Convert any audio format to WAV using ffmpeg. Returns path to WAV file."""
    if input_path is None:
        return None
    ext = os.path.splitext(input_path)[1].lower()
    if ext == ".wav":
        return input_path
    tmp = tempfile.NamedTemporaryFile(suffix=".wav", delete=False)
    tmp.close()
    subprocess.run(
        ["ffmpeg", "-y", "-i", input_path, "-ar", "22050", "-ac", "1", tmp.name],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
    )
    return tmp.name


def load_audio(audio_path):
    """Load any audio format (WAV, MP3, OGG, OPUS, M4A, etc). Returns (signal, sample_rate)."""
    wav_path = convert_to_wav(audio_path)
    audio, sr = librosa.load(wav_path, sr=None)
    return audio, sr


def calculate_spectral_features(audio, sr):
    """Extract MFCC (13 coeff) + spectral centroid, flatness, rolloff."""
    S = np.abs(librosa.stft(audio))
    spectral_centroid = librosa.feature.spectral_centroid(S=S)[0]
    spectral_flatness = librosa.feature.spectral_flatness(S=S)[0]
    spectral_rolloff  = librosa.feature.spectral_rolloff(S=S)[0]
    mfcc      = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=13)
    mfcc_mean = np.mean(mfcc, axis=1)
    return spectral_centroid, spectral_flatness, spectral_rolloff, mfcc_mean


def compare_audio_features(f1, f2):
    """Return spectral score, MFCC cosine distance and per-feature diffs."""
    min_c = min(len(f1[0]), len(f2[0]))
    min_f = min(len(f1[1]), len(f2[1]))
    min_r = min(len(f1[2]), len(f2[2]))
    centroid_diff = np.mean(np.abs(f1[0][:min_c] - f2[0][:min_c]))
    flatness_diff = np.mean(np.abs(f1[1][:min_f] - f2[1][:min_f]))
    rolloff_diff  = np.mean(np.abs(f1[2][:min_r] - f2[2][:min_r]))
    cosine_sim    = np.dot(f1[3], f2[3]) / (np.linalg.norm(f1[3]) * np.linalg.norm(f2[3]))
    mfcc_distance = 1 - cosine_sim
    spectral_score = centroid_diff + flatness_diff + rolloff_diff
    return spectral_score, mfcc_distance, centroid_diff, flatness_diff, rolloff_diff


def generate_pdf_report(path1, path2, spectral_score, mfcc_distance,
                        centroid_diff, flatness_diff, rolloff_diff, verdict):
    """Save a PDF report with all metrics."""
    pdf = FPDF()
    pdf.add_page()
    pdf.set_font("Arial", "B", 16)
    pdf.cell(0, 10, "Audio Similarity Report", ln=True, align="C")
    pdf.ln(5)
    pdf.set_font("Arial", size=12)
    pdf.cell(0, 10, "Audio 1: " + path1, ln=True)
    pdf.cell(0, 10, "Audio 2: " + path2, ln=True)
    pdf.ln(5)
    pdf.set_font("Arial", "B", 12)
    pdf.cell(0, 10, "Results:", ln=True)
    pdf.set_font("Arial", size=12)
    pdf.cell(0, 10, "Verdict: " + verdict, ln=True)
    pdf.cell(0, 10, "Spectral Score: " + str(round(spectral_score, 4)), ln=True)
    pdf.cell(0, 10, "MFCC Cosine Distance: " + str(round(mfcc_distance, 4)) + "  (< 0.15 = same person)", ln=True)
    pdf.cell(0, 10, "Spectral Centroid diff: " + str(round(centroid_diff, 4)), ln=True)
    pdf.cell(0, 10, "Spectral Flatness diff: " + str(round(flatness_diff, 6)), ln=True)
    pdf.cell(0, 10, "Spectral Rolloff diff:  " + str(round(rolloff_diff, 4)), ln=True)
    pdf.output("similarity_report.pdf")
    print("PDF report saved as similarity_report.pdf")


def determine_similarity(audio_path_1, audio_path_2):
    """Compare two audio files. Returns (verdict, mfcc_distance, spectrogram_path)."""
    audio_1, sr_1 = load_audio(audio_path_1)
    audio_2, sr_2 = load_audio(audio_path_2)
    if sr_1 != sr_2:
        print("Warning: different sample rates (" + str(sr_1) + " vs " + str(sr_2) + " Hz).")
    f1 = calculate_spectral_features(audio_1, sr_1)
    f2 = calculate_spectral_features(audio_2, sr_2)
    result = compare_audio_features(f1, f2)
    spectral_score = result[0]
    mfcc_distance  = result[1]
    centroid_diff  = result[2]
    flatness_diff  = result[3]
    rolloff_diff   = result[4]
    if mfcc_distance < 0.10:
        verdict = "Same person.  (High confidence)"
    elif mfcc_distance < 0.25:
        verdict = "Uncertain — possibly different persons.  (Low confidence)"
    else:
        verdict = "Different persons.  (High confidence)"
    print(verdict)
    fig, axes = plt.subplots(2, 1, figsize=(12, 6))
    librosa.display.specshow(
        librosa.amplitude_to_db(np.abs(librosa.stft(audio_1)), ref=np.max),
        sr=sr_1, x_axis="time", y_axis="log", ax=axes[0])
    axes[0].set_title("Spectrogram - Audio 1")
    fig.colorbar(axes[0].collections[0], ax=axes[0], format="%+2.0f dB")
    librosa.display.specshow(
        librosa.amplitude_to_db(np.abs(librosa.stft(audio_2)), ref=np.max),
        sr=sr_2, x_axis="time", y_axis="log", ax=axes[1])
    axes[1].set_title("Spectrogram - Audio 2")
    fig.colorbar(axes[1].collections[0], ax=axes[1], format="%+2.0f dB")
    plt.tight_layout()
    spectrogram_path = "spectrogram_output.png"
    plt.savefig(spectrogram_path, dpi=150, bbox_inches="tight")
    plt.close()
    print("Spectrogram saved as " + spectrogram_path)
    print("MFCC Distance:          " + str(round(mfcc_distance, 4)) + "  (threshold 0.15)")
    print("Spectral Centroid diff: " + str(round(centroid_diff, 4)))
    print("Spectral Flatness diff: " + str(round(flatness_diff, 6)))
    print("Spectral Rolloff diff:  " + str(round(rolloff_diff, 4)))
    generate_pdf_report(audio_path_1, audio_path_2, spectral_score, mfcc_distance,
                        centroid_diff, flatness_diff, rolloff_diff, verdict)
    return verdict, mfcc_distance, spectrogram_path


ModuleNotFoundError: No module named 'gradio'

## Download sample files
Downloads the included synthetic WAV files from the repository. Skip this cell if you already uploaded your own audio files.


In [ ]:
# Download sample audio files from GitHub (required for Colab)
import urllib.request, os

base = "https://github.com/DaniAngel79/voice-similarity-checker/raw/main/"
files = ["audio_path_1.wav", "audio_path_2.wav"]

for fname in files:
    if not os.path.exists(fname):
        urllib.request.urlretrieve(base + fname, fname)
        print("Downloaded: " + fname)
    else:
        print("Already exists: " + fname)


## Demo - Sample audio files
Runs with the two included synthetic WAV files.
Expected output: verdict + spectrogram image + PDF report.


In [ ]:
verdict, mfcc_dist, spec_path = determine_similarity("audio_path_1.wav", "audio_path_2.wav")
print("Verdict      : " + verdict)
print("MFCC Distance: " + str(round(mfcc_dist, 4)))


## Interactive UI - Gradio
Launches a local web interface at `http://localhost:7860`.
Upload **any two audio files** (WAV or MP3) to compare them.

> On **Google Colab** a public share link is printed automatically (`share=True`).


In [ ]:
def gradio_interface(audio1, audio2):
    if audio1 is None or audio2 is None:
        return "Please upload both audio files.", None
    verdict, mfcc_dist, spectrogram_path = determine_similarity(audio1, audio2)
    result_text = verdict + "  |  MFCC Distance: " + str(round(mfcc_dist, 4))
    return result_text, spectrogram_path


interface = gr.Interface(
    fn=gradio_interface,
    inputs=[
        gr.Audio(type="filepath", label="Audio 1"),
        gr.Audio(type="filepath", label="Audio 2")
    ],
    outputs=[
        gr.Textbox(label="Result"),
        gr.Image(label="Spectrograms")
    ],
    title="Voice Similarity Checker",
    description="Upload two voice recordings to check if they belong to the same person. Supports WAV, MP3, OGG, OPUS, M4A, FLAC and more.",
    examples=[["audio_path_1.wav", "audio_path_2.wav"]]
)

interface.launch(share=True)
